# 🏁 Maillon 1 — Python : ingestion des résultats

Vous recevez `../donnees/resultats.csv`, l'export brut d'un championnat de F1 (5 courses, 10 pilotes) :

```
course;pilote;ecurie;position;meilleur_tour;statut
Bahrein;VERSTAPPEN;Red Bull;1;1:33.614;ARRIVE
Bahrein;HAMILTON;Mercedes;;;ABANDON
```

**Votre mission** : produire `../02-java/courses_propres.csv` au format du **CONTRAT 1**, que le maillon Java consommera :

```
course;pilote;ecurie;position;temps_tour
Bahrein;VERSTAPPEN;Red Bull;1;93.614
Bahrein;HAMILTON;Mercedes;0;
```

Deux transformations seulement :
1. le chronomètre `1:33.614` devient un nombre de secondes `93.614` ;
2. un abandon devient la **position 0** et un temps **vide** (la colonne `statut` disparaît).

**Méthode** : complétez les trois fonctions, exécutez la cellule de tests jusqu'au 4/4, puis lancez la
cellule de production qui écrit le vrai fichier pour le maillon Java.

## 1. Convertir un chronomètre en secondes

In [77]:
def temps_en_secondes(texte):
    """'1:33.996' -> 93.996 (float, arrondi à 3 décimales).
    Une chaîne vide ou ne contenant que des espaces -> None."""
    if not texte or not texte.strip():
        return None
    try:
        minutes, secondes = texte.split(':')
        total_secondes = int(minutes) * 60 + float(secondes)
        return round(total_secondes, 3)
    except ValueError:
        return None
    except Exception:
        return None
    pass

## 2. Lire le fichier brut

In [78]:
def lire_resultats(chemin):
    """Lit le CSV brut et renvoie une liste de dictionnaires :
    {"course": str, "pilote": str, "ecurie": str, "position": int, "temps_tour": float|None}
    - position : l'entier du CSV, ou 0 si le statut est ABANDON
    - temps_tour : converti avec temps_en_secondes (None si absent)
    La ligne d'en-tête ne doit pas figurer dans le résultat."""
    

    resultats = [] #On initialise une liste vide pour stocker les résultats
    with open(chemin, 'r', encoding="utf-8") as f:
        next(f) #On saute la première ligne qui est l'en-tête
        for ligne in f: #On parcourt chaque ligne du fichier
            ligne = ligne.strip() #On enlève les espaces en début et fin de ligne
            if not ligne: #Si la ligne est vide, on passe à la suivante
                continue 
            if ";" in ligne: #On détermine le séparateur utilisé dans la ligne (point-virgule)
                sep = ";"
            else: # ou (virgule
                sep = "," 
            elements = [e.strip() for e in ligne.split(sep)] #On sépare la ligne en éléments et on enlève les espaces autour de chaque élément
            if len(elements) < 4: #Si la ligne ne contient pas au moins 4 éléments, on passe à la suivante
                continue 

            course, pilote, ecurie, position_str = elements[:4] #On récupère les 4 premiers éléments de la ligne
            temps_tour_str = elements[4] if len(elements) > 4 else "" #On récupère le 5ème élément si présent, sinon on met une chaîne vide

            position = int(position_str) if position_str.isdigit() else 0 #On convertit la position en entier, ou on met 0 si ce n'est pas un chiffre
            temps_tour = temps_en_secondes(temps_tour_str) #On convertit le temps du tour en secondes avec la fonction temps_en_secondes

            resultats.append({
                "course": course,
                "pilote": pilote,
                "ecurie": ecurie,
                "position": position,
                "temps_tour": temps_tour
            }) #On ajoute un dictionnaire avec les informations de la ligne à la liste des résultats
    return resultats
    pass

## 3. Écrire le fichier du contrat 1

In [79]:
def ecrire_courses_propres(chemin, lignes):
    """Écrit le CONTRAT 1 : en-tête course;pilote;ecurie;position;temps_tour
    - temps_tour est écrit avec 3 décimales, ou vide si None
    - les lignes sont écrites dans l'ordre reçu"""
    with open(chemin, 'w', encoding='utf-8') as f:
        f.write("course;pilote;ecurie;position;temps_tour\n") #On écrit l'en-tête du fichier
        for ligne in lignes: #On parcourt chaque ligne de la liste de lignes
            temps_tour = f"{ligne['temps_tour']:.3f}" if ligne['temps_tour'] is not None else "" #On formate le temps du tour avec 3 décimales ou on met une chaîne vide si None
            f.write(f"{ligne['course']};{ligne['pilote']};{ligne['ecurie']};{ligne['position']};{temps_tour}\n") #On écrit les informations de la ligne dans le fichier
    pass

## ✅ Tests

In [80]:
# ✅ Tests — exécutez cette cellule (ne pas modifier)
import os, tempfile

_resultats = {}

def _egal(obtenu, attendu):
    assert obtenu == attendu, f"attendu {attendu!r}, obtenu {obtenu!r}"

def verifier(nom, controle):
    try:
        controle()
        _resultats[nom] = True
        print(f"✅ {nom}")
    except Exception as err:
        _resultats[nom] = False
        print(f"❌ {nom} → {type(err).__name__} : {err}")

CSV_TEST = (
    "course;pilote;ecurie;position;meilleur_tour;statut\n"
    "Bahrein;VERSTAPPEN;Red Bull;1;1:33.996;ARRIVE\n"
    "Bahrein;LECLERC;Ferrari;2;1:34.211;ARRIVE\n"
    "Bahrein;HAMILTON;Mercedes;;;ABANDON\n"
)

def _fichier(contenu=""):
    chemin = os.path.join(tempfile.mkdtemp(), "f.csv")
    with open(chemin, "w", encoding="utf-8") as f:
        f.write(contenu)
    return chemin

def _test_temps():
    _egal(temps_en_secondes("1:33.996"), 93.996)
    _egal(temps_en_secondes("0:59.500"), 59.5)
    _egal(temps_en_secondes("2:00.000"), 120.0)
    _egal(temps_en_secondes(""), None)
    _egal(temps_en_secondes("   "), None)

def _test_lire():
    lignes = lire_resultats(_fichier(CSV_TEST))
    _egal(len(lignes), 3)
    _egal(lignes[0], {"course": "Bahrein", "pilote": "VERSTAPPEN", "ecurie": "Red Bull",
                      "position": 1, "temps_tour": 93.996})
    _egal(lignes[2]["position"], 0)
    _egal(lignes[2]["temps_tour"], None)
    _egal(type(lignes[1]["position"]).__name__, "int")

def _test_ecrire():
    chemin = _fichier("ancien contenu\n")
    ecrire_courses_propres(chemin, [
        {"course": "Bahrein", "pilote": "VERSTAPPEN", "ecurie": "Red Bull", "position": 1, "temps_tour": 93.996},
        {"course": "Bahrein", "pilote": "HAMILTON", "ecurie": "Mercedes", "position": 0, "temps_tour": None},
    ])
    with open(chemin, encoding="utf-8") as f:
        contenu = f.read()
    _egal(contenu,
          "course;pilote;ecurie;position;temps_tour\n"
          "Bahrein;VERSTAPPEN;Red Bull;1;93.996\n"
          "Bahrein;HAMILTON;Mercedes;0;\n")

def _test_chaine():
    entree = _fichier(CSV_TEST)
    sortie = _fichier()
    ecrire_courses_propres(sortie, lire_resultats(entree))
    with open(sortie, encoding="utf-8") as f:
        lignes = f.read().splitlines()
    _egal(len(lignes), 4)
    _egal(lignes[1], "Bahrein;VERSTAPPEN;Red Bull;1;93.996")
    _egal(lignes[3], "Bahrein;HAMILTON;Mercedes;0;")

verifier("1. temps_en_secondes", _test_temps)
verifier("2. lire_resultats", _test_lire)
verifier("3. ecrire_courses_propres", _test_ecrire)
verifier("4. chaîne complète", _test_chaine)

reussis = sum(1 for ok in _resultats.values() if ok)
print(f"\n{reussis} / {len(_resultats)} tests réussis" + (" — maillon Python validé 🎉" if reussis == 4 else ""))

✅ 1. temps_en_secondes
✅ 2. lire_resultats
✅ 3. ecrire_courses_propres
✅ 4. chaîne complète

4 / 4 tests réussis — maillon Python validé 🎉


## 🏁 Production du fichier pour le maillon Java

In [81]:
# 🏁 Production du fichier réel (à exécuter une fois les 4 tests au vert)
ENTREE = "../donnees/resultats.csv"
SORTIE = "../02-java/courses_propres.csv"

lignes = lire_resultats(ENTREE)
ecrire_courses_propres(SORTIE, lignes)

print(f"{len(lignes)} lignes lues, fichier écrit : {SORTIE}\n")
for ligne in lignes[:5]:
    print(ligne)
abandons = [l for l in lignes if l["position"] == 0]
print(f"\n{len(abandons)} abandons :", [l["pilote"] for l in abandons])

50 lignes lues, fichier écrit : ../02-java/courses_propres.csv

{'course': 'Bahrein', 'pilote': 'VERSTAPPEN', 'ecurie': 'Red Bull', 'position': 1, 'temps_tour': 93.614}
{'course': 'Bahrein', 'pilote': 'LECLERC', 'ecurie': 'Ferrari', 'position': 2, 'temps_tour': 93.637}
{'course': 'Bahrein', 'pilote': 'SAINZ', 'ecurie': 'Ferrari', 'position': 3, 'temps_tour': 94.507}
{'course': 'Bahrein', 'pilote': 'NORRIS', 'ecurie': 'McLaren', 'position': 4, 'temps_tour': 92.614}
{'course': 'Bahrein', 'pilote': 'RUSSELL', 'ecurie': 'Mercedes', 'position': 5, 'temps_tour': 92.916}

4 abandons : ['HAMILTON', 'PEREZ', 'ALONSO', 'RUSSELL']
